Website Summary from URL using Ollama and BeautifySoap


In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display


In [2]:
from openai import OpenAI

In [3]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [4]:
# to pretend to be a real user
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [5]:
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [6]:
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

In [7]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [8]:
def summarize(url):
    website = Website(url)
    response = ollama.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [9]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [10]:
display_summary("https://anthropic.com")

### Anthropic Website Summary

#### Mission and Purpose
Anthropic aims to secure the benefits of AI while mitigating its risks. As a public benefit corporation, they prioritize humanity's long-term well-being.

#### Latest News and Announcements
- **Claude Opus 4.6**: Released, offering the world's most powerful model for coding, agents, and professional work.
- **Project Glasswing**: Securing critical software for the AI era.
- **NASA's Perseverance Rover**: Claude assisted in its journey with an AI-assisted drive on Mars.

#### Products and Services
- **Claude**: A platform to facilitate genuine conversations without ads or sponsored content.
- **Claude Opus**, **Sonnet**, **Haiku**: Models for specific tasks.
- **Claude Code**: For coding, with enterprise, security, and custom plans available.
- **Claude Console**: Offers tools and services for developers.

#### Partnerships and Research
- **Anthropic's Economic Index**: Examines the economic implications of AI.
- **Alignment Science** and **Anthropic Academy**: Provide education and resources on AI safety and development.

### Cookie Policy

* The website uses cookies for security, performance analysis, ads personalization, and tracking.
* Users can customize or reject cookie settings as desired.

In [11]:
display_summary("https://ollama.com/")

**Summary of Ollama Website**
=====================================

Ollama is a platform that enables users to build, automate, and run AI-powered models with ease.

### Key Features

* OpenClaw: A cloud-based model automation tool with open models powered by Ollama.
* Cloud Models: Access faster, larger models on datacenter-grade hardware.
* Pricing:
	+ Free (with an Ollama account): Access basic cloud models and run local or cloud-based tasks.
	+ Pro ($20/month or $200/year): Run 3 cloud models at a time with 50x more usage.
	+ Max ($100/month): Run 10 cloud models at a time with 5x more usage.

### Announcements

Ollama Launches OpenClaw: Automate your work and scale with the power of Ollama's cloud-based models.